# 06 — Camada de pontos da RAIS e lista de CEPs a localizar

**Objetivo:** duas saídas, geradas depois da etapa 05:

1. **Camada de pontos da RAIS** (`04_entrega\rais_estab_pontos.gpkg`, camada `rais_pontos`): **uma feição por registro da RAIS** (todos os anos) cujo CEP tem ponto (classes **A**, **B** e **G**). Traz todas as colunas da RAIS (menos as de `pontos.colunas_excluir` no `config.yaml`) + `classe_cep`, `motivo_cep`, `ponto_metodo`, `n_partes` e `linha_rais`.
   - **A**: ponto confiável (via curta, um logradouro) · **B**: aproximado (via longa ou várias vias) · **G**: grande usuário (etapa 04; precisão em `ponto_metodo`). **Filtre por `classe_cep`.**
   - Classes C, D e E não têm ponto: estão em `residuo_rais_nao_espacializada.parquet` (etapa 05).
   - Também copia `ceps_espacializados.gpkg` (linhas e pontos por CEP) para `04_entrega`.
2. **Excel `ceps_a_localizar.xlsx`** (`03_processamento\06_pontos_e_pendencias`): o que localizar à mão, em ordem de vínculos, com sugestões de vias por nome para os CEPs órfãos. As sugestões **por localização** (CNEFE) estão na etapa 07.

**Estrutura:** 1 Preparação · 2 Funções · 3 Carregar entradas · 4 Camada de pontos · 5 Exportar camada · 6 Sugestões de vias · 7 Excel · 8 Resumo

## 1. Preparação e configuração

In [ ]:
# --- Preparação padrão: acha a raiz do projeto, lê o config.yaml e liga o log (tela + arquivo) -------------
import sys                                            # para o Python achar as funções comuns (scripts/src)
from pathlib import Path                              # caminhos de arquivos
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())   # sobe pastas até achar o config.yaml
sys.path.insert(0, str(RAIZ / "scripts"))             # libera o "from src import ..."
from src import config, registro                      # funções comuns do padrão de projetos
cfg = config.carregar(RAIZ)                           # lê o config.yaml (caminhos e parâmetros)
ETAPA = "06_pontos_e_pendencias"            # nome desta etapa (= nome do notebook)
log = registro.iniciar_log(ETAPA, cfg)                # log na tela e em 03_processamento/logs/
PASTA_SAIDA = cfg.pasta_etapa(ETAPA)                  # 03_processamento/<ETAPA> (criada se não existir)
import shutil                         # cópia de arquivos
import numpy as np                    # contas com colunas
import pandas as pd                   # tabelas
import pyogrio                        # leitura/escrita de arquivos geográficos
import geopandas as gpd               # tabelas com geometria
from rapidfuzz import fuzz, process   # comparação de textos "parecidos"
from src import fontes                # arquivos editados (prioridade editado -> bruto)

# --- Entradas (config.yaml) ---------------------------------------------------------------------------
ARQ_RAIS_CLASS = cfg.caminho("processamento.rais_classificada")    # etapa 05
ARQ_DIAG = cfg.caminho("processamento.diagnostico")                # etapa 05
ARQ_PONTOS_CEP = cfg.caminho("processamento.ceps_espacializados")  # etapa 05 (camada ceps_pontos)
ARQ_BASE = fontes.caminho_editado(cfg, "logradouros_cep")          # base oficial editada
CAMADA_BASE = cfg.camada("editados.logradouros_cep")
ARQ_PONTE = cfg.caminho("processamento.ponte")                     # universo de CEPs dos Correios (etapa 02)
ARQ_ORFAOS = cfg.caminho("processamento.orfaos")

# --- Saídas ---------------------------------------------------------------------------------------------------
ARQ_PONTOS_RAIS = cfg.caminho("entrega.rais_pontos")               # 04_entrega/rais_estab_pontos.gpkg
CAMADA_PONTOS_RAIS = cfg.camada("entrega.rais_pontos")
ARQ_CEPS_ENTREGA = cfg.caminho("entrega.ceps_espacializados")      # cópia das linhas/pontos por CEP na entrega
ARQ_EXCEL = PASTA_SAIDA / "ceps_a_localizar.xlsx"                  # lista de trabalho (regerável)

# --- Parâmetros ------------------------------------------------------------------------------------------------
COLUNAS_EXCLUIR = cfg.parametro("pontos.colunas_excluir")          # colunas da RAIS fora da camada de pontos
CLASSES_PONTO = ["A", "B", "G"]                                    # classes com ponto
CLASSES_LOCALIZAR = ["C", "D"]                                     # classes da aba principal do Excel
LIMIAR_SUGESTAO = cfg.parametro("pontos.limiar_sugestao")          # similaridade mínima do nome para sugerir via
N_SUGESTOES = cfg.parametro("pontos.n_sugestoes")                  # sugestões por CEP
MAX_IDS_SUGESTAO = 5                                               # ids de segmento mostrados por sugestão
CRS_ESPERADO = cfg.projeto("crs_metrico")

log.info("Configuração carregada!")
for _nome, _arq in [("RAIS classificada", ARQ_RAIS_CLASS), ("Diagnóstico", ARQ_DIAG), ("Pontos dos CEPs", ARQ_PONTOS_CEP),
                    ("Base oficial", ARQ_BASE), ("Ponte (Correios)", ARQ_PONTE), ("Órfãos (Correios)", ARQ_ORFAOS)]:
    log.info("%s: %s | existe: %s", _nome, _arq.name, _arq.exists())

## 2. Funções auxiliares
Vêm de `scripts/src` (comuns a todos os projetos).

In [ ]:
from src.validacao import validar_tabela          # bloco padrão de validação
from src.texto import normalizar, tirar_tipo      # normalização de nomes de logradouro
from src.geo import para_gpkg                     # tipos compatíveis com GeoPackage
from src.planilha import formatar_excel           # formatação das abas do Excel
log.info("Funções auxiliares carregadas!")

## 3. Carregar entradas

In [ ]:
log.info("Carregando as saídas da Etapa 05...")
rais_cls = pd.read_parquet(ARQ_RAIS_CLASS)
rais_cls["linha_rais"] = np.arange(len(rais_cls), dtype="int64")          # posição do registro na RAIS classificada (rastreio)
for col in ["cep_chave", "classe_cep", "motivo_cep", "espacializavel"]:
    assert col in rais_cls.columns, f"Coluna '{col}' não existe na RAIS classificada (rode a Etapa 05 atualizada)"
log.info("RAIS classificada: %d linhas | anos: %s", len(rais_cls), sorted(rais_cls["ano"].unique().tolist()))
diag = pd.read_parquet(ARQ_DIAG)
log.info("Diagnóstico: %d CEPs", len(diag))

log.info("Carregando os pontos dos CEPs...")
pontos_cep = pyogrio.read_dataframe(ARQ_PONTOS_CEP, layer="ceps_pontos", columns=["cep", "ponto_metodo", "n_partes"])
assert pontos_cep.crs is not None and pontos_cep.crs.to_string() == CRS_ESPERADO, f"CRS inesperado: {pontos_cep.crs}"
validar_tabela("Pontos dos CEPs", pontos_cep, chave=["cep"], obrigatorias=["cep"])

log.info("Carregando a base oficial (sem geometria) e o universo dos Correios...")
base = pyogrio.read_dataframe(ARQ_BASE, layer=CAMADA_BASE, read_geometry=False,
                              columns=["id", "tipo", "nome", "loteamento", "cep", "ceps_candidatos"])
for col in ["tipo", "nome", "loteamento", "cep", "ceps_candidatos"]:
    base[col] = base[col].astype(object).where(base[col].notna(), "").astype(str).str.strip()
ponte = pd.read_parquet(ARQ_PONTE, columns=["cep", "logradouro_correios", "bairro_correios"])
orfaos_e1 = pd.read_csv(ARQ_ORFAOS, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
universo = pd.concat([
    ponte.groupby("cep", as_index=False).agg(logradouro_orig=("logradouro_correios", "first"), bairro_orig=("bairro_correios", "first")),
    orfaos_e1[["cep", "logradouro_orig", "bairro_orig"]]]).drop_duplicates("cep")
ceps_base = set(base.loc[base["cep"] != "", "cep"].str.split("|").explode().str.replace(r"\D", "", regex=True))
log.info("Base: %d segmentos | CEPs distintos na base: %d | universo dos Correios: %d CEPs", len(base), len(ceps_base), len(universo))
log.info("Entradas carregadas!")

## 4. Camada de pontos da RAIS
Cada registro da RAIS **espacializável** (CEP de classe A ou B) herda o ponto do seu CEP. Não há duplicação: um registro = um ponto. Pontos de registros com o mesmo CEP ficam **empilhados** no mesmo lugar (use `classe_cep` e os atributos para filtrar/agregar).

In [ ]:
log.info("Montando a camada de pontos da RAIS...")
sel = rais_cls[rais_cls["classe_cep"].isin(CLASSES_PONTO)].copy()
n_sel = len(sel)
pts = pd.DataFrame(pontos_cep).rename(columns={"cep": "cep_chave"})
sel = sel.merge(pts, on="cep_chave", how="left", validate="many_to_one")
sem_ponto = int(sel["geometry"].isna().sum())
(log.info if sem_ponto == 0 else log.error)("Registros A/B sem ponto do CEP: %d", sem_ponto)
sel = sel[sel["geometry"].notna()].copy()

# --- Colunas: RAIS + classificação + ponto; tira as que você não quer na camada ---------------------------------------------
colunas_rais = [c for c in rais_cls.columns if c not in ["cep_chave", "classe_cep", "motivo_cep", "espacializavel", "linha_rais"] + COLUNAS_EXCLUIR]
colunas_final = ["linha_rais"] + colunas_rais + ["classe_cep", "motivo_cep", "ponto_metodo", "n_partes", "geometry"]
colunas_final = [c for c in colunas_final if c in sel.columns]
rais_pontos = gpd.GeoDataFrame(sel[colunas_final], geometry="geometry", crs=pontos_cep.crs).reset_index(drop=True)
log.info("Colunas na camada (%d): %s", len(colunas_final) - 1, [c for c in colunas_final if c != "geometry"])

# --- Validação: registros, perda, vínculos e contagem por ano --------------------------------------------------------------------
ok_n = len(rais_pontos) == int(rais_cls["classe_cep"].isin(CLASSES_PONTO).sum())
(log.info if ok_n else log.error)("[%s] pontos = registros de classe %s na RAIS (%d)", "OK" if ok_n else "ERRO", CLASSES_PONTO, len(rais_pontos))
if "qtd_vinc_ativos" in rais_pontos.columns:
    v_pts = int(rais_pontos["qtd_vinc_ativos"].sum()); v_cls = int(rais_cls.loc[rais_cls["classe_cep"].isin(CLASSES_PONTO), "qtd_vinc_ativos"].sum())
    log.info("[%s] vínculos ativos na camada = %d (esperado %d)", "OK" if v_pts == v_cls else "ERRO", v_pts, v_cls)
tabela_ano = (rais_cls.assign(com_ponto=rais_cls["classe_cep"].isin(CLASSES_PONTO))
              .groupby("ano").agg(registros=("linha_rais", "size"), com_ponto=("com_ponto", "sum")))
tabela_ano["pct_com_ponto"] = (tabela_ano["com_ponto"] / tabela_ano["registros"] * 100).round(1)
log.info("Registros com ponto por ano:\n%s", tabela_ano.to_string())
log.info("Por classe:\n%s", rais_pontos["classe_cep"].value_counts().to_string())
validar_tabela("Camada de pontos da RAIS", rais_pontos, chave=["linha_rais"], obrigatorias=["linha_rais", "classe_cep"], linhas_entrada=n_sel)
log.info("Camada de pontos montada!")

## 5. Exportar a camada de pontos
Esta camada é **regenerada a cada execução** (é derivada da base + RAIS), então o arquivo anterior é substituído.

In [ ]:
log.info("Gravando a camada de pontos...")
if ARQ_PONTOS_RAIS.exists():
    ARQ_PONTOS_RAIS.unlink()                                  # recomeça do zero
para_gpkg(rais_pontos).to_file(ARQ_PONTOS_RAIS, layer=CAMADA_PONTOS_RAIS, driver="GPKG", engine="pyogrio")
log.info("GeoPackage salvo: %s | camada %s | %d feições | CRS %s", ARQ_PONTOS_RAIS.name, CAMADA_PONTOS_RAIS, len(rais_pontos), rais_pontos.crs)
ARQ_CEPS_ENTREGA.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(ARQ_PONTOS_CEP, ARQ_CEPS_ENTREGA)                # linhas e pontos por CEP também na entrega
log.info("Cópia de %s em %s", ARQ_PONTOS_CEP.name, ARQ_CEPS_ENTREGA.parent)
log.info("Exportação da camada concluída!")

## 6. Sugestões de vias para os CEPs a localizar
Para cada CEP **órfão da RAIS (classe D)**, procura na base vias de **nome parecido** (sem o tipo, mesma normalização). Ordena pela similaridade do nome e, no empate, do bairro com o loteamento. Cada sugestão traz os **ids dos segmentos** (campo `id` da base, para localizar no QGIS) e o **CEP atual** da via, pois ela pode já ter outro CEP.

In [ ]:
log.info("Procurando vias parecidas para os CEPs órfãos...")
# --- Vias da base (uma linha por tipo + nome + loteamento) --------------------------------------------------------------------------
nomeados = base[base["nome"] != ""].copy()
vias = (nomeados.groupby(["tipo", "nome", "loteamento"], dropna=False)
        .agg(ids=("id", list),
             ceps_atuais=("cep", lambda s: "|".join(sorted(set(x for x in s if x)))))
        .reset_index())
vias["nome_key"] = normalizar(vias["nome"])
vias["lote_key"] = normalizar(vias["loteamento"])
vias = vias[vias["nome_key"] != ""].reset_index(drop=True)
log.info("Vias distintas na base: %d (sem CEP: %d)", len(vias), int((vias["ceps_atuais"] == "").sum()))

# --- CEPs órfãos da RAIS (classe D) -----------------------------------------------------------------------------------------------------
alvos = diag[diag["classe"] == "D"].copy()
alvos["nome_key"] = normalizar(alvos["logradouro_correios"]).map(tirar_tipo)
alvos["bairro_key"] = normalizar(alvos["bairro_correios"])
alvos = alvos[alvos["nome_key"] != ""]
log.info("CEPs órfãos (classe D) a sugerir: %d", len(alvos))

sugestoes_largo = pd.DataFrame({"cep": alvos["cep"].values})
if len(alvos) and len(vias):
    q_nomes = alvos["nome_key"].drop_duplicates().tolist()
    c_nomes = vias["nome_key"].drop_duplicates().tolist()
    matriz = process.cdist(q_nomes, c_nomes, scorer=fuzz.token_sort_ratio, score_cutoff=LIMIAR_SUGESTAO, workers=-1)
    pos_vias = vias.groupby("nome_key").indices                       # nome normalizado -> posições em "vias"
    achados = []
    for i, q in enumerate(q_nomes):                                    # laço curto (só nomes distintos), sem iterrows
        for j in np.flatnonzero(matriz[i] > 0):
            for pos in pos_vias[c_nomes[j]]:
                achados.append((q, pos, float(matriz[i, j])))
    sug = pd.DataFrame(achados, columns=["nome_key", "via_pos", "score_nome"])
    log.info("Pares nome × via acima de %d: %d", LIMIAR_SUGESTAO, len(sug))
    if len(sug):
        sug = sug.merge(alvos[["cep", "nome_key", "bairro_key"]], on="nome_key", how="inner")
        sug = sug.join(vias[["tipo", "nome", "loteamento", "lote_key", "ids", "ceps_atuais"]], on="via_pos")
        pares_b = sug[["bairro_key", "lote_key"]].drop_duplicates()
        pares_b["score_bairro"] = [fuzz.ratio(a, b) for a, b in zip(pares_b["bairro_key"], pares_b["lote_key"])]
        sug = sug.merge(pares_b, on=["bairro_key", "lote_key"], how="left")
        sug = sug.sort_values(["cep", "score_nome", "score_bairro"], ascending=[True, False, False])
        sug["ordem"] = sug.groupby("cep").cumcount() + 1
        top = sug[sug["ordem"] <= N_SUGESTOES].copy()
        top["via_txt"] = top["tipo"] + " " + top["nome"] + np.where(top["loteamento"] != "", " (" + top["loteamento"] + ")", "")
        top["ids_txt"] = top["ids"].map(lambda l: "|".join(str(i) for i in l[:MAX_IDS_SUGESTAO]) + (" ..." if len(l) > MAX_IDS_SUGESTAO else ""))
        top["cep_atual_txt"] = np.where(top["ceps_atuais"] != "", top["ceps_atuais"], "sem CEP")
        largo = sugestoes_largo.set_index("cep")
        for k in range(1, N_SUGESTOES + 1):
            t = top[top["ordem"] == k].set_index("cep")
            largo[f"sug{k}_via"] = t["via_txt"]
            largo[f"sug{k}_ids"] = t["ids_txt"]
            largo[f"sug{k}_cep_atual"] = t["cep_atual_txt"]
            largo[f"sug{k}_score_nome"] = t["score_nome"].round(1)
            largo[f"sug{k}_score_bairro"] = t["score_bairro"].round(1)
        sugestoes_largo = largo.reset_index()
com_sug = int(sugestoes_largo["sug1_via"].notna().sum()) if "sug1_via" in sugestoes_largo else 0
log.info("CEPs órfãos com ao menos 1 sugestão: %d de %d (%.1f%%)", com_sug, len(alvos), com_sug / len(alvos) * 100 if len(alvos) else 0)
log.info("Sugestões prontas!")

## 7. Excel "CEPs a localizar"
Abas:
1. **CEPs a localizar** — classes C e D da RAIS, em ordem de vínculos (ano de referência), com `pct_acum_vinc_ref` (quanto da fila já cobre), `vias_candidatas` (classe C) e até 3 sugestões de vias da base (classe D);
2. **CEPs fora dos Correios (E)** — CEPs da RAIS que não estão na lista dos Correios (inválidos, prefixo fora de 12, sufixo especial...), para verificar ou descartar;
3. **CEPs da base fora dos Correios** — CEPs que **você preencheu na base** e que **não constam nos Correios** (possível erro de digitação);
4. **Órfãos fora da RAIS** — CEPs dos Correios sem segmento na base que não aparecem na RAIS (prioridade baixa);
5. **Resumo** — contagens por classe.

Não há coluna "resolvido": ao corrigir a base e rodar a Etapa 05 de novo, o CEP resolvido muda de classe e sai desta lista.

In [ ]:
log.info("Montando o Excel...")
lista = diag[diag["classe"].isin(CLASSES_LOCALIZAR)].copy()
lista = lista.sort_values(["vinc_ativos_ref", "vinc_ativos_soma_anos", "n_estab_soma_anos"], ascending=False).reset_index(drop=True)
lista.insert(0, "posicao", np.arange(1, len(lista) + 1))
total_vinc = lista["vinc_ativos_ref"].sum()
lista["pct_acum_vinc_ref"] = (lista["vinc_ativos_ref"].cumsum() / total_vinc * 100).round(1) if total_vinc else 0.0
lista["vias_candidatas"] = np.where(lista["classe"] == "C", lista["vias_ambiguas_exemplos"], None)
lista = lista.merge(sugestoes_largo, on="cep", how="left")
colunas_sug = [c for c in lista.columns if c.startswith("sug")]
aba1 = lista[["posicao", "cep", "classe", "motivo", "logradouro_correios", "bairro_correios", "n_estab_ref",
              "vinc_ativos_ref", "pct_acum_vinc_ref", "n_estab_soma_anos", "vinc_ativos_soma_anos", "vias_candidatas"] + colunas_sug]

aba2 = (diag[diag["classe"] == "E"].sort_values("vinc_ativos_ref", ascending=False)
        [["cep", "motivo", "n_estab_ref", "vinc_ativos_ref", "n_estab_soma_anos", "vinc_ativos_soma_anos"]])
aba3 = (diag[diag["cep_fora_correios"]].sort_values("vinc_ativos_ref", ascending=False)
        [["cep", "classe", "n_segmentos", "n_logradouros", "metodos", "n_estab_ref", "vinc_ativos_ref", "vinc_ativos_soma_anos"]])
ceps_rais = set(diag["cep"])
aba4 = (universo[~universo["cep"].isin(ceps_base) & ~universo["cep"].isin(ceps_rais)]
        [["cep", "logradouro_orig", "bairro_orig"]].sort_values("logradouro_orig"))
aba5 = (diag.groupby("classe").agg(n_ceps=("cep", "size"), n_estab_ref=("n_estab_ref", "sum"), vinc_ativos_ref=("vinc_ativos_ref", "sum"),
                                   vinc_ativos_soma_anos=("vinc_ativos_soma_anos", "sum")).reset_index())

with pd.ExcelWriter(ARQ_EXCEL, engine="openpyxl") as escritor:
    for nome_aba, tabela in [("CEPs a localizar", aba1), ("CEPs fora dos Correios (E)", aba2),
                             ("CEPs da base fora dos Correios", aba3), ("Órfãos fora da RAIS", aba4), ("Resumo", aba5)]:
        tabela.to_excel(escritor, sheet_name=nome_aba, index=False)
        formatar_excel(escritor, nome_aba)
log.info("Excel salvo: %s | linhas: a localizar %d, fora dos Correios (E) %d, base fora dos Correios %d, órfãos fora da RAIS %d",
         ARQ_EXCEL.name, len(aba1), len(aba2), len(aba3), len(aba4))
validar_tabela("Aba 'CEPs a localizar'", aba1, chave=["cep"], obrigatorias=["cep", "classe"])
log.info("Excel pronto!")

## 8. Resumo

In [ ]:
log.info("Resumo final...")
n_total = len(rais_cls); n_pts = len(rais_pontos)
log.info("Registros da RAIS (todos os anos): %d | com ponto: %d (%.1f%%) | sem ponto (C/D/E): %d", n_total, n_pts, n_pts / n_total * 100, n_total - n_pts)
log.info("Pendências: %d CEPs classes %s | %d CEPs classe E | %d CEPs da base fora dos Correios", len(aba1), CLASSES_LOCALIZAR, len(aba2), len(aba3))
if len(aba1):
    for corte in (50, 80, 90):
        n = int((aba1["pct_acum_vinc_ref"] < corte).sum() + 1)
        log.info("  %d primeiros CEPs da lista cobrem ~%d%% dos vínculos pendentes (C/D)", min(n, len(aba1)), corte)
log.info("Próximo passo do ciclo: corrigir 'cep' na base oficial (use também a etapa 07) e rodar python executar_tudo.py --ciclo.")
log.info("Etapa 06 concluída!")